# Qwen3-4B QLoRA — data v2 đầy đủ

Upload `sft_raw_history_v2_all.zip` thành **private Kaggle Dataset**, attach vào notebook này.
Bật **Internet** và **GPU T4 x2**. Code clone từ public repo, nhánh `test`.
Giữ 4.397 train / 472 validation, không cắt hoặc bỏ mẫu; context tối đa 12.288 token.
Chạy từ trên xuống: cài thư viện → clone/code + data → kiểm tra → smoke mẫu dài nhất → full train → đóng gói.
Chưa xác nhận VRAM/tốc độ bằng GPU khi chuẩn bị notebook. Smoke thất bại thì không chạy full.
Validation loss không phải điểm tool-calling. Không đưa validation vào train.
T4 attention: expand KV heads explicitly; fused SDPA only, no quadratic math fallback. A small attention forward/backward probe runs before loading model weights.


In [ ]:
import sys, subprocess
packages = ['transformers==4.57.1', 'peft==0.17.1', 'accelerate==1.10.1', 'bitsandbytes==0.48.1', 'liger-kernel==0.6.2', 'librosa>=0.11,<0.12', 'safetensors']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, json
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONUNBUFFERED'] = '1'
WORK = Path('/kaggle/working/lifemate-qwen-v2')
PUBLIC_REPO = 'https://github.com/tuong1901/zac2026-lifemate.git'
if not (WORK / '.git').is_dir():
    assert not WORK.exists() or not any(WORK.iterdir()), 'Choose an empty WORK directory'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'test', PUBLIC_REPO, str(WORK)], check=True)
else:
    assert subprocess.check_output(['git', '-C', str(WORK), 'remote', 'get-url', 'origin'], text=True).strip() == PUBLIC_REPO
    assert subprocess.check_output(['git', '-C', str(WORK), 'branch', '--show-current'], text=True).strip() == 'test'
    changes = subprocess.check_output(['git', '-C', str(WORK), 'status', '--porcelain'], text=True).strip()
    if changes:
        print('Keeping local code edits; pull skipped:', changes)
    else:
        subprocess.run(['git', '-C', str(WORK), 'pull', '--ff-only', 'origin', 'test'], check=True)
os.chdir(WORK)
assert (WORK / 'scripts/train_qwen_v2.py').is_file(), 'Update the public test branch first'
print('Code commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
import torch
assert torch.cuda.is_available(), 'Enable GPU'
NPROC = torch.cuda.device_count()
assert NPROC == 2, 'Choose T4 x2, then restart the kernel before importing torch'
for i in range(NPROC):
    print(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1), 'GiB')

In [ ]:
# Override this if several copies of the data are attached.
DATA_DIR = None  # Example: Path('/kaggle/input/datasets/<owner>/<dataset>')
INPUT = Path('/kaggle/input')
if DATA_DIR is None:
    candidates = []
    for report_path in INPUT.rglob('report.json'):
        try:
            report = json.loads(report_path.read_text())
            if (report.get('package') == 'all_lengths' and report.get('input_mode') == 'raw'
                and (report_path.parent / 'train.jsonl').is_file()
                and (report_path.parent / 'validation.jsonl').is_file()):
                candidates.append(report_path.parent)
        except (OSError, ValueError):
            pass
    if candidates:
        assert len(candidates) == 1, f'Set DATA_DIR explicitly: {candidates}'
        DATA_DIR = candidates[0]
    else:
        packages = list(INPUT.rglob('sft_raw_history_v2_all.zip'))
        assert len(packages) == 1, f'Attach the full v2 Dataset, found zip files: {packages}'
        DATA_DIR = Path('/kaggle/working/sft_raw_history_v2_all')
        DATA_DIR.mkdir(exist_ok=True)
        with zipfile.ZipFile(packages[0]) as archive:
            for member in archive.infolist():
                assert (DATA_DIR / member.filename).resolve().is_relative_to(DATA_DIR.resolve())
            archive.extractall(DATA_DIR)
for name, expected in [('train.jsonl', 4397), ('validation.jsonl', 472)]:
    path = DATA_DIR / name
    assert path.is_file(), path
    count = sum(bool(line.strip()) for line in path.open(encoding='utf-8'))
    assert count == expected, f'{name}: {count}; expected full data v2 ({expected})'
    print(name, count)
print('Data:', DATA_DIR)

In [ ]:
MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
MAX_LENGTH = 12288
OUT = Path('/kaggle/working/qwen_v2_training')
SMOKE_OUT = Path('/kaggle/working/qwen_v2_smoke_t4_fused_v1')
RESUME = False  # True only if OUT contains full checkpoint-N + run_config.json.
common = ['--data-dir', str(DATA_DIR), '--model', MODEL, '--max-length', str(MAX_LENGTH),
          '--epochs', '2', '--lr', '1e-4', '--rank', '16', '--grad-accum', '8', '--save-steps', '50']
def launch(extra):
    command = [sys.executable, '-u', '-m', 'torch.distributed.run', '--standalone',
               f'--nproc_per_node={NPROC}', 'scripts/train_qwen_v2.py'] + common + extra
    subprocess.run(command, cwd=WORK, check=True)
# If this fails/OOMs, stop here and keep the error. Do not lower max-length to silently drop data.
receipt = SMOKE_OUT / 'smoke_passed.json'
if not receipt.is_file():
    launch(['--out', str(SMOKE_OUT), '--smoke'])
print(json.loads(receipt.read_text()))

In [ ]:
# Full training begins only after a matching longest-sequence smoke run succeeds.
extra = ['--out', str(OUT), '--smoke-receipt', str(receipt)]
if RESUME:
    extra += ['--resume', 'auto']
launch(extra)
print(json.loads((OUT / 'metrics.json').read_text()))
print('Adapter:', OUT / 'final_adapter')

In [ ]:
import shutil
FINAL = OUT / 'final_adapter'
assert (FINAL / 'adapter_config.json').is_file()
assert (FINAL / 'adapter_model.safetensors').is_file()
adapter_zip = shutil.make_archive('/kaggle/working/qwen_v2_final_adapter', 'zip', FINAL)
print('Download adapter:', adapter_zip)
# This may be much larger, but contains optimizer/scheduler/RNG state for resume.
recovery_zip = shutil.make_archive('/kaggle/working/qwen_v2_training_recovery', 'zip', OUT)
print('Download resume backup:', recovery_zip)
print('Use Save Version and download outputs before the runtime expires.')

`final_adapter` là LoRA, cần base `Qwen/Qwen3-4B-Instruct-2507` để inference.
Không trỏ adapter trực tiếp vào script merge ASR. Giữ `qwen_v2_training_recovery.zip` nếu muốn resume đúng trạng thái.
Đánh giá tool-calling sau train phải dùng cùng RAW_SYSTEM_PROMPT và cấu trúc raw-history v2.
Bản này chọn adapter cuối, chưa tự chọn checkpoint theo điểm thực thi.
Không dùng validation loss để công bố điểm public hoặc kết luận mô hình tốt hơn baseline.